# Análise Exploratória das Variáveis SIM para o Modelo Dimensional

**Objetivo:** Identificar os valores distintos de cada variável do SIM que será usada nas dimensões do star schema, verificando:
- Valores únicos (categorias reais)
- Valores ausentes (nulos/vazios)
- Compatibilidade entre anos (2014-2023)
- Distribuição de frequência

**Variáveis-alvo por dimensão:**
| Dimensão | Coluna(s) SIM |
|---|---|
| Dim_Tempo | DTOBITO |
| Dim_Municipio | CODMUNOCOR, CODMUNRES |
| Dim_FaixaEtaria | IDADE |
| Dim_RacaCor | RACACOR |
| Dim_LocalOcorrencia | LOCOCOR |
| Dim_SituacaoGestacionalObito | TPMORTEOCO |
| Dim_TipoParto | PARTO |
| Dim_MomentoObitoParto | OBITOPARTO |
| Dim_TipoGravidez | GRAVIDEZ |
| Dim_SemanaGestacao | SEMAGESTAC |
| Dim_CID | CAUSABAS, CAUSAMAT, CAUSABAS_O |
| Dim_EstabelecimentoSaude | CODESTAB |
| Atributo direto na fato | SEXO, ASSISTMED |

In [1]:
import pandas as pd
import os
import glob

print("pandas version:", pd.__version__)

pandas version: 3.0.3


In [2]:
# Configuração: pasta dos dados SIM
pasta_sim = "../arquivos/SIM"

# Lista todos os CSVs anuais, ordenados
arquivos = sorted(glob.glob(os.path.join(pasta_sim, "dados_*.csv")))
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(f"  {os.path.basename(a)}")

Arquivos encontrados: 10
  dados_2014.csv
  dados_2015.csv
  dados_2016.csv
  dados_2017.csv
  dados_2018.csv
  dados_2019.csv
  dados_2020.csv
  dados_2021.csv
  dados_2022.csv
  dados_2023.csv


In [3]:
# Colunas de interesse para o modelo dimensional
COLUNAS_INTERESSE = [
    'DTOBITO',       # Dim_Tempo
    'SEXO',          # Atributo direto na fato (normalizado)
    'IDADE',         # Dim_FaixaEtaria
    'RACACOR',       # Dim_RacaCor
    'LOCOCOR',       # Dim_LocalOcorrencia
    'TPMORTEOCO',    # Dim_SituacaoGestacionalObito
    'PARTO',         # Dim_TipoParto
    'OBITOPARTO',    # Dim_MomentoObitoParto
    'GRAVIDEZ',      # Dim_TipoGravidez
    'SEMAGESTAC',    # Dim_SemanaGestacao
    'CAUSABAS',      # Dim_CID (causa básica)
    'CAUSAMAT',      # Dim_CID (causa materna)
    'CAUSABAS_O',    # Dim_CID (causa básica original)
    'CODMUNOCOR',    # Dim_Municipio (ocorrência)
    'CODMUNRES',     # Dim_Municipio (residência)
    'CODESTAB',      # Dim_EstabelecimentoSaude
    'ASSISTMED',     # recebeu_assistencia_medica
]

In [4]:
# Função para ler e consolidar todos os anos com apenas as colunas de interesse
def carregar_dados_sim(arquivos, colunas):
    """Lê CSVs do SIM, normaliza colunas para maiúsculo e retorna DataFrame consolidado."""
    frames = []
    for arquivo in arquivos:
        ano = os.path.basename(arquivo).replace('dados_', '').replace('.csv', '')
        print(f"Lendo {os.path.basename(arquivo)}...", end=' ')
        df_ano = pd.read_csv(arquivo, sep=';', encoding='latin1', dtype=str, low_memory=False)
        df_ano.columns = df_ano.columns.str.upper()
        
        # Verifica quais colunas de interesse existem neste arquivo
        cols_presentes = [c for c in colunas if c in df_ano.columns]
        cols_ausentes = [c for c in colunas if c not in df_ano.columns]
        
        if cols_ausentes:
            print(f"(faltam: {cols_ausentes})", end=' ')
        
        df_ano = df_ano[cols_presentes].copy()
        df_ano['_ano'] = ano
        frames.append(df_ano)
        print(f"{len(df_ano)} registros")
    
    df = pd.concat(frames, ignore_index=True)
    print(f"\nTotal consolidado: {len(df)} registros ({len(frames)} anos)")
    return df

df_sim = carregar_dados_sim(arquivos, COLUNAS_INTERESSE)

Lendo dados_2014.csv... 1227039 registros
Lendo dados_2015.csv... 1264175 registros
Lendo dados_2016.csv... 1309774 registros
Lendo dados_2017.csv... 1312663 registros
Lendo dados_2018.csv... 1316719 registros
Lendo dados_2019.csv... 1349801 registros
Lendo dados_2020.csv... 1556824 registros
Lendo dados_2021.csv... 1832649 registros
Lendo dados_2022.csv... 1544266 registros
Lendo dados_2023.csv... 1465610 registros

Total consolidado: 14179520 registros (10 anos)


## 1. SEXO — Valores Originais no SIM

Mapeamento definido no plano:
| Original (SIM) | Normalizado |
|---|---|
| 'F', '2' | 'F' |
| 'M', '1' | 'M' |
| 'I', '0', '9', outros | 'I' |

Vamos verificar os valores reais presentes.

In [5]:
# --- SEXO ---
print("=" * 60)
print("SEXO - Valores originais no SIM")
print("=" * 60)

sexo_counts = df_sim['SEXO'].value_counts(dropna=False).sort_index()
print(sexo_counts.to_string())
print(f"\nTotal: {sexo_counts.sum()}")
print(f"Nulos: {df_sim['SEXO'].isna().sum()}")

# Aplicar a normalização proposta
def normaliza_sexo(valor):
    if pd.isna(valor):
        return 'I'
    valor = str(valor).strip().upper()
    if valor in ('F', '2'):
        return 'F'
    elif valor in ('M', '1'):
        return 'M'
    else:
        return 'I'

df_sim['sexo_normalizado'] = df_sim['SEXO'].apply(normaliza_sexo)
print("\n--- Distribuição após normalização ---")
print(df_sim['sexo_normalizado'].value_counts().to_string())

SEXO - Valores originais no SIM
SEXO
0       6292
1    7891122
2    6282106

Total: 14179520
Nulos: 0

--- Distribuição após normalização ---
sexo_normalizado
M    7891122
F    6282106
I       6292


In [6]:
# Verificar consistência do SEXO por ano
print("SEXO por ano (valores originais):\n")
print(pd.crosstab(df_sim['_ano'], df_sim['SEXO'], margins=True).to_string())

SEXO por ano (valores originais):

SEXO     0        1        2       All
_ano                                  
2014   755   693922   532362   1227039
2015   675   709117   554383   1264175
2016   573   736842   572359   1309774
2017   621   734469   577573   1312663
2018   646   733616   582457   1316719
2019   557   745519   603725   1349801
2020   630   874167   682027   1556824
2021   683  1015350   816616   1832649
2022   626   844920   698720   1544266
2023   526   803200   661884   1465610
All   6292  7891122  6282106  14179520


## 2. IDADE — Faixa Etária

O campo IDADE no SIM tem 3 dígitos:
- 1º dígito: unidade de medida (`4` = anos)
- 2º e 3º dígitos: valor

Faixas previstas: 10-14, 15-19, 20-24, 25-29, 30-34, 35-39, 40-44, 45-49

O filtro de mortalidade materna usa `IDADE BETWEEN '410' AND '449'` (10 a 49 anos).

In [7]:
# --- IDADE ---
print("=" * 60)
print("IDADE - Valores originais no SIM")
print("=" * 60)

# Mostrar distribuição completa
idade_counts = df_sim['IDADE'].value_counts(dropna=False).sort_index()
print(f"Total de valores distintos: {len(idade_counts)}")
print(f"\nPrimeiros 30 valores:")
print(idade_counts.head(30).to_string())
print(f"\nÚltimos 30 valores:")
print(idade_counts.tail(30).to_string())

# Verificar os primeiros dígitos (unidade de medida)
print("\n\n--- Primeiro dígito do IDADE (unidade de medida) ---")
primeiro_digito = df_sim['IDADE'].dropna().str[0].value_counts().sort_index()
print(primeiro_digito.to_string())

IDADE - Valores originais no SIM
Total de valores distintos: 257

Primeiros 30 valores:
IDADE
001    2137
002     641
003     479
004     250
005    1833
006     204
007     196
008     219
009     164
010    1955
011     172
012     219
013     217
014     197
015    1303
016     195
017     235
018     231
019     180
020    1782
021     220
022     299
023     305
024     233
025     807
026     237
027     255
028     271
029     200
030    2339

Últimos 30 valores:
IDADE
502    16194
503    11891
504     8398
505     5774
506     3996
507     2941
508     2097
509     1439
510     1023
511      739
512      497
513      374
514      234
515      162
516      121
517       64
518       34
519       15
520       10
521       11
522        8
523        7
524        5
525        2
526        3
527        1
529        1
531        1
533        1
999    27140


--- Primeiro dígito do IDADE (unidade de medida) ---
IDADE
0       27198
1       54593
2      162841
3      102565
4    1369714

In [8]:
# IDADE: analisar apenas registros com 1º dígito = 4 (medida em anos)
# e verificar a distribuição de idade (2 últimos dígitos)
df_idade_anos = df_sim[df_sim['IDADE'].str[0] == '4'].copy()
df_idade_anos['idade_valor'] = df_idade_anos['IDADE'].str[1:3].astype(int)

print("Distribuição de idade (anos) - apenas registros com unidade=4:\n")
print(df_idade_anos['idade_valor'].value_counts().sort_index().to_string())

# Faixas etárias conforme plano
faixas = [
    (10, 14, '10-14'),
    (15, 19, '15-19'),
    (20, 24, '20-24'),
    (25, 29, '25-29'),
    (30, 34, '30-34'),
    (35, 39, '35-39'),
    (40, 44, '40-44'),
    (45, 49, '45-49'),
    (50, 200, '50+'),
]

def classificar_faixa(idade):
    if pd.isna(idade):
        return None
    for min_id, max_id, nome in faixas:
        if min_id <= idade <= max_id:
            return nome
    return 'DESCONHECIDA'

df_idade_anos['faixa_etaria'] = df_idade_anos['idade_valor'].apply(classificar_faixa)
print("\n--- Distribuição por faixa etária ---")
print(df_idade_anos['faixa_etaria'].value_counts().to_string())

Distribuição de idade (anos) - apenas registros com unidade=4:

idade_valor
0         35
1      24983
2      13925
3       9987
4       8451
5       7202
6       6522
7       6098
8       5830
9       6017
10      6218
11      6556
12      7584
13      9653
14     13984
15     20332
16     29097
17     38227
18     45192
19     49890
20     53302
21     53905
22     53823
23     53531
24     52290
25     52925
26     52817
27     53209
28     53963
29     54123
30     55714
31     56967
32     59181
33     61678
34     63758
35     67201
36     68941
37     72077
38     74803
39     77377
40     81755
41     84473
42     87603
43     90672
44     95207
45     99592
46    104217
47    109803
48    116369
49    122172
50    131254
51    138380
52    145280
53    153572
54    162021
55    170341
56    179897
57    190292
58    198015
59    206727
60    215020
61    222400
62    230917
63    240062
64    246361
65    253958
66    261849
67    267228
68    271819
69    276311
70    280816
7

## 3. RACACOR — Raça/Cor

| Código | Descrição |
|---|---|
| 1 | Branca |
| 2 | Preta |
| 3 | Amarela |
| 4 | Parda |
| 5 | Indígena |

In [9]:
# --- RACACOR ---
print("=" * 60)
print("RACACOR - Raça/Cor")
print("=" * 60)

racacor_counts = df_sim['RACACOR'].value_counts(dropna=False).sort_index()
print(racacor_counts.to_string())
print(f"\nNulos: {df_sim['RACACOR'].isna().sum()}")
print(f"Valores distintos: {racacor_counts.index.tolist()}")

RACACOR - Raça/Cor
RACACOR
1      7190546
2      1139838
3        82161
4      5304863
5        45281
9            5
NaN     416826

Nulos: 416826
Valores distintos: ['1', '2', '3', '4', '5', '9', nan]


## 4. LOCOCOR — Local de Ocorrência

| Código | Descrição |
|---|---|
| 1 | Hospital |
| 2 | Outros est. saúde |
| 3 | Domicílio |
| 4 | Via pública |
| 5 | Outros |
| 6 | Aldeia indígena |
| 9 | Ignorado |

In [10]:
# --- LOCOCOR ---
print("=" * 60)
print("LOCOCOR - Local de Ocorrência")
print("=" * 60)

lococor_counts = df_sim['LOCOCOR'].value_counts(dropna=False).sort_index()
print(lococor_counts.to_string())
print(f"\nNulos: {df_sim['LOCOCOR'].isna().sum()}")
print(f"Valores distintos: {sorted(lococor_counts.index[~pd.isna(lococor_counts.index)].astype(str).tolist())}")

LOCOCOR - Local de Ocorrência
LOCOCOR
1      9478168
2       865393
3      2811714
4       554208
5       457735
6          908
9        11391
NaN          3

Nulos: 3
Valores distintos: ['1', '2', '3', '4', '5', '6', '9']


## 5. TPMORTEOCO — Situação Gestacional do Óbito

| Código | Descrição |
|---|---|
| 1 | Na gravidez |
| 2 | No parto |
| 3 | No abortamento |
| 4 | Até 42 dias pós-parto |
| 5 | 43d a 1 ano pós-parto |
| 8 | Não ocorreu nestes períodos |
| 9 | Ignorado |

In [11]:
# --- TPMORTEOCO ---
print("=" * 60)
print("TPMORTEOCO - Situação Gestacional do Óbito")
print("=" * 60)

tpmorteoco_counts = df_sim['TPMORTEOCO'].value_counts(dropna=False).sort_index()
print(tpmorteoco_counts.to_string())
print(f"\nNulos: {df_sim['TPMORTEOCO'].isna().sum()}")
print(f"Valores distintos: {sorted(tpmorteoco_counts.index[~pd.isna(tpmorteoco_counts.index)].astype(str).tolist())}")

TPMORTEOCO - Situação Gestacional do Óbito
TPMORTEOCO
1          6673
2          1896
3           781
4         12182
5          6794
6            29
7             4
8        733654
9        149670
NaN    13267837

Nulos: 13267837
Valores distintos: ['1', '2', '3', '4', '5', '6', '7', '8', '9']


## 6. PARTO — Tipo de Parto

| Código | Descrição |
|---|---|
| 1 | Vaginal |
| 2 | Cesáreo |
| 9 | Ignorado |

In [12]:
# --- PARTO ---
print("=" * 60)
print("PARTO - Tipo de Parto")
print("=" * 60)

parto_counts = df_sim['PARTO'].value_counts(dropna=False).sort_index()
print(parto_counts.to_string())
print(f"\nNulos: {df_sim['PARTO'].isna().sum()}")
print(f"Valores distintos: {sorted(parto_counts.index[~pd.isna(parto_counts.index)].astype(str).tolist())}")

PARTO - Tipo de Parto
PARTO
1        155907
2        158774
9          4880
NaN    13859959

Nulos: 13859959
Valores distintos: ['1', '2', '9']


## 7. OBITOPARTO — Momento do Óbito em relação ao Parto

| Código | Descrição |
|---|---|
| 1 | Antes |
| 2 | Durante |
| 3 | Depois |
| 9 | Ignorado |

In [13]:
# --- OBITOPARTO ---
print("=" * 60)
print("OBITOPARTO - Momento do Óbito em relação ao Parto")
print("=" * 60)

obitoparto_counts = df_sim['OBITOPARTO'].value_counts(dropna=False).sort_index()
print(obitoparto_counts.to_string())
print(f"\nNulos: {df_sim['OBITOPARTO'].isna().sum()}")
print(f"Valores distintos: {sorted(obitoparto_counts.index[~pd.isna(obitoparto_counts.index)].astype(str).tolist())}")

OBITOPARTO - Momento do Óbito em relação ao Parto
OBITOPARTO
1             6
2             7
3        309970
9          6934
NaN    13862603

Nulos: 13862603
Valores distintos: ['1', '2', '3', '9']


## 8. GRAVIDEZ — Tipo de Gravidez

| Código | Descrição |
|---|---|
| 1 | Única |
| 2 | Dupla |
| 3 | Tripla e mais |
| 9 | Ignorada |

In [14]:
# --- GRAVIDEZ ---
print("=" * 60)
print("GRAVIDEZ - Tipo de Gravidez")
print("=" * 60)

gravidez_counts = df_sim['GRAVIDEZ'].value_counts(dropna=False).sort_index()
print(gravidez_counts.to_string())
print(f"\nNulos: {df_sim['GRAVIDEZ'].isna().sum()}")
print(f"Valores distintos: {sorted(gravidez_counts.index[~pd.isna(gravidez_counts.index)].astype(str).tolist())}")

GRAVIDEZ - Tipo de Gravidez
GRAVIDEZ
1        286634
2         28843
3          1906
9          3783
NaN    13858354

Nulos: 13858354
Valores distintos: ['1', '2', '3', '9']


## 9. SEMAGESTAC — Semana de Gestação

| Faixa | Semanas |
|---|---|
| 0-21 | 0 a 21 |
| 22-27 | 22 a 27 |
| 28-31 | 28 a 31 |
| 32-36 | 32 a 36 |
| 37-41 | 37 a 41 |
| 42+ | 42 ou mais |

In [15]:
# --- SEMAGESTAC ---
print("=" * 60)
print("SEMAGESTAC - Semana de Gestação")
print("=" * 60)

semagestac_counts = df_sim['SEMAGESTAC'].value_counts(dropna=False).sort_index()
print(semagestac_counts.to_string())
print(f"\nNulos: {df_sim['SEMAGESTAC'].isna().sum()}")
print(f"Valores distintos: {sorted(semagestac_counts.index[~pd.isna(semagestac_counts.index)].astype(str).tolist())}")

SEMAGESTAC - Semana de Gestação
SEMAGESTAC
0          2192
1          2393
10           73
11           42
12           76
13           20
14           38
15           50
16          151
17          165
18          352
19          786
2          1602
20         2891
21         4507
22         8846
23        11652
24        15196
25        14234
26        15932
27        13946
28        14673
29        11150
3          1051
30        11128
31         9636
32        11119
33         9967
34        10930
35        11225
36        14121
37        17165
38        22535
39        25440
4           550
40        18688
41         6555
42         1771
43          131
44           53
45            1
5           294
6           205
7           131
8           114
9           272
99         6302
NaN    13879169

Nulos: 13879169
Valores distintos: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', 

## 10. CID — Causas (Role-Playing Dimension)

Três colunas de CID que apontam para a mesma Dim_CID:
- **CAUSABAS**: Causa básica (NOT NULL na fato)
- **CAUSAMAT**: Causa materna
- **CAUSABAS_O**: Causa básica original

In [16]:
# --- CAUSABAS (causa básica) ---
print("=" * 60)
print("CAUSABAS - Causa Básica (NOT NULL)")
print("=" * 60)

causabas_counts = df_sim['CAUSABAS'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causabas_counts)}")
print(f"Nulos: {df_sim['CAUSABAS'].isna().sum()}")
print(f"\nTop 20 CIDs mais frequentes:")
print(causabas_counts.head(20).to_string())
print(f"\nBottom 10 CIDs menos frequentes:")
print(causabas_counts.tail(10).to_string())

CAUSABAS - Causa Básica (NOT NULL)
Total de CIDs distintos: 7717
Nulos: 0

Top 20 CIDs mais frequentes:
CAUSABAS
A010       6
A011       1
A014       2
A020      53
A021     193
A022      14
A028       4
A029      20
A031       1
A038       1
A039      10
A040      40
A041      15
A042      14
A043       7
A044      56
A045       4
A046       6
A047    1785
A048     335

Bottom 10 CIDs menos frequentes:
CAUSABAS
Y870      71
Y871     460
Y872     565
Y880      32
Y881      25
Y882      26
Y883     535
Y890      14
Y891       2
Y899    1021


In [17]:
# --- CAUSAMAT (causa materna) ---
print("=" * 60)
print("CAUSAMAT - Causa Materna")
print("=" * 60)

causamat_counts = df_sim['CAUSAMAT'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causamat_counts)}")
print(f"Nulos: {df_sim['CAUSAMAT'].isna().sum()}")
# Muitos nulos são esperados, pois nem todo óbito tem causa materna
nao_nulos = df_sim['CAUSAMAT'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 20 CIDs maternos mais frequentes:")
print(causamat_counts.head(20).to_string())

CAUSAMAT - Causa Materna
Total de CIDs distintos: 11
Nulos: 14179243
Não nulos: 277 (0.0%)

Top 20 CIDs maternos mais frequentes:
CAUSAMAT
O930          85
O931           6
O932           1
O933           4
O934          17
O935           5
O936          47
O937          95
O938           8
O939           9
NaN     14179243


In [18]:
# --- CAUSABAS_O (causa básica original) ---
print("=" * 60)
print("CAUSABAS_O - Causa Básica Original")
print("=" * 60)

causabas_o_counts = df_sim['CAUSABAS_O'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causabas_o_counts)}")
print(f"Nulos: {df_sim['CAUSABAS_O'].isna().sum()}")
nao_nulos = df_sim['CAUSABAS_O'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 20 CIDs originais mais frequentes:")
print(causabas_o_counts.head(20).to_string())

CAUSABAS_O - Causa Básica Original
Total de CIDs distintos: 9004
Nulos: 12788
Não nulos: 14166732 (99.9%)

Top 20 CIDs originais mais frequentes:
CAUSABAS_O
A001      1
A009      3
A010      7
A011      1
A014      2
A02       1
A020     47
A021    188
A022     13
A028      5
A029     18
A03       1
A031      1
A038      1
A039      8
A04       5
A040     38
A041     13
A042     14
A043      7


## 11. DTOBITO — Data do Óbito (Dim_Tempo)

Verificar amplitude de datas, anos cobertos e valores malformados.

In [27]:
# --- DTOBITO ---
print("=" * 60)
print("DTOBITO - Data do Óbito")
print("=" * 60)

# DTOBITO está no formato DDMMYYYY (ex: 27092014 = 27/09/2014)
# Primeiro, verificar formatos das datas
print("Amostra de 20 valores de DTOBITO:")
print(df_sim['DTOBITO'].dropna().sample(20, random_state=42).to_string())

# Extrair ano (posições 4-7 do formato DDMMYYYY)
df_sim['ano_dtobito'] = df_sim['DTOBITO'].str[4:8]
print(f"\n--- Distribuição de anos ---")
ano_counts = df_sim['ano_dtobito'].value_counts(dropna=False).sort_index()
print(ano_counts.to_string())

# Verificar valores com formato inesperado
print(f"\n--- Valores com comprimento diferente de 8 ---")
tamanhos = df_sim['DTOBITO'].dropna().str.len().value_counts().sort_index()
print(f"Distribuição de comprimentos: {tamanhos.to_dict()}")

# Verificar nulos
print(f"\nNulos em DTOBITO: {df_sim['DTOBITO'].isna().sum()}")

# Extrair data completa no formato ISO (YYYY-MM-DD)
df_sim['data_iso'] = df_sim['DTOBITO'].apply(
    lambda x: f"{x[4:8]}-{x[2:4]}-{x[0:2]}" if pd.notna(x) and len(str(x)) == 8 else None
)
print(f"\n--- Amostra de datas convertidas para ISO ---")
print(df_sim['data_iso'].dropna().sample(10, random_state=42).to_string())

DTOBITO - Data do Óbito
Amostra de 20 valores de DTOBITO:
163475      27092014
8889121     05082020
913293      27082014
1467892     29062015
9768297     21062021
4669879     18112017
10229676    11032021
5505092     28032018
9192700     20072020
6602583     07042019
10859091    06042021
6456357     03022019
4535735     07082017
1698275     28052015
6677224     25032019
6393357     24122018
4151438     02122017
2601416     29092016
12298722    18072022
1365230     25072015

--- Distribuição de anos ---
ano_dtobito
2014    1227039
2015    1264175
2016    1309774
2017    1312663
2018    1316719
2019    1349801
2020    1556824
2021    1832649
2022    1544266
2023    1465610

--- Valores com comprimento diferente de 8 ---
Distribuição de comprimentos: {8: 14179520}

Nulos em DTOBITO: 0

--- Amostra de datas convertidas para ISO ---
163475      2014-09-27
8889121     2020-08-05
913293      2014-08-27
1467892     2015-06-29
9768297     2021-06-21
4669879     2017-11-18
10229676    2021-03-11

## 12. CODMUNOCOR e CODMUNRES — Municípios

A mesma Dim_Municipio é usada como role-playing (ocorrência e residência).

In [20]:
# --- CODMUNOCOR (Município de Ocorrência) ---
print("=" * 60)
print("CODMUNOCOR - Município de Ocorrência")
print("=" * 60)

mun_ocor_counts = df_sim['CODMUNOCOR'].value_counts(dropna=False).sort_index()
print(f"Total de municípios distintos: {len(mun_ocor_counts)}")
print(f"Nulos: {df_sim['CODMUNOCOR'].isna().sum()}")
print(f"\nTop 10 municípios:")
print(mun_ocor_counts.head(10).to_string())
print(f"\nÚltimos 5:")
print(mun_ocor_counts.tail(5).to_string())

# Verificar comprimento do código
print(f"\n--- Comprimento do código ---")
print(df_sim['CODMUNOCOR'].dropna().str.len().value_counts().sort_index().to_string())

CODMUNOCOR - Município de Ocorrência
Total de municípios distintos: 5591
Nulos: 0

Top 10 municípios:
CODMUNOCOR
110000        5
110001      827
110002     5939
110003      173
110004    12080
110005      606
110006      536
110007      147
110008      337
110009     1002

Últimos 5:
CODMUNOCOR
522200       655
522205       315
522220       197
522230       160
530010    158884

--- Comprimento do código ---
CODMUNOCOR
6    14179520


In [21]:
# --- CODMUNRES (Município de Residência) ---
print("=" * 60)
print("CODMUNRES - Município de Residência")
print("=" * 60)

mun_res_counts = df_sim['CODMUNRES'].value_counts(dropna=False).sort_index()
print(f"Total de municípios distintos: {len(mun_res_counts)}")
print(f"Nulos: {df_sim['CODMUNRES'].isna().sum()}")
print(f"\nTop 10 municípios:")
print(mun_res_counts.head(10).to_string())

# Comparar cobertura
print(f"\n--- Comparação Ocorrência vs Residência ---")
print(f"Municípios apenas em CODMUNOCOR: {len(set(mun_ocor_counts.index) - set(mun_res_counts.index))}")
print(f"Municípios apenas em CODMUNRES: {len(set(mun_res_counts.index) - set(mun_ocor_counts.index))}")
print(f"Municípios em ambos: {len(set(mun_ocor_counts.index) & set(mun_res_counts.index))}")

CODMUNRES - Município de Residência
Total de municípios distintos: 5595
Nulos: 0

Top 10 municípios:
CODMUNRES
110000     168
110001    1456
110002    5811
110003     355
110004    5369
110005    1076
110006    1039
110007     379
110008     662
110009    1733

--- Comparação Ocorrência vs Residência ---
Municípios apenas em CODMUNOCOR: 0
Municípios apenas em CODMUNRES: 4
Municípios em ambos: 5591


## 13. CODESTAB — Estabelecimento de Saúde (CNES)

Código do estabelecimento para vincular com Dim_EstabelecimentoSaude.

In [22]:
# --- CODESTAB ---
print("=" * 60)
print("CODESTAB - Estabelecimento de Saúde")
print("=" * 60)

codestab_counts = df_sim['CODESTAB'].value_counts(dropna=False).sort_index()
print(f"Total de estabelecimentos distintos: {len(codestab_counts)}")
print(f"Nulos: {df_sim['CODESTAB'].isna().sum()}")
nao_nulos = df_sim['CODESTAB'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 10 estabelecimentos:")
print(codestab_counts.head(10).to_string())
print(f"\nComprimento do código:")
print(df_sim['CODESTAB'].dropna().str.len().value_counts().sort_index().to_string())

CODESTAB - Estabelecimento de Saúde
Total de estabelecimentos distintos: 23553
Nulos: 3836211
Não nulos: 10343309 (72.9%)

Top 10 estabelecimentos:
CODESTAB
0000001      1
0000010      1
0000011      2
0000013      1
0000014      5
0000017      1
0000018    682
0000019    456
0000020      1
0000021      9

Comprimento do código:
CODESTAB
7    10343309


## 14. ASSISTMED — Assistência Médica

| Código | Descrição |
|---|---|
| 1 | Sim |
| 2 (ou ausente) | Não / Ignorado |

In [23]:
# --- ASSISTMED ---
print("=" * 60)
print("ASSISTMED - Recebeu Assistência Médica")
print("=" * 60)

assistmed_counts = df_sim['ASSISTMED'].value_counts(dropna=False).sort_index()
print(assistmed_counts.to_string())
print(f"\nNulos: {df_sim['ASSISTMED'].isna().sum()}")
print(f"Valores distintos: {sorted(assistmed_counts.index[~pd.isna(assistmed_counts.index)].astype(str).tolist())}")

ASSISTMED - Recebeu Assistência Médica
ASSISTMED
1      7551205
2      1429007
9       845036
NaN    4354272

Nulos: 4354272
Valores distintos: ['1', '2', '9']


## 15. APLICAÇÃO DO FILTRO DE MORTALIDADE MATERNA

Validar o filtro definido no plano:

```sql
WHERE (SEXO = 'F' AND IDADE BETWEEN '410' AND '449')
   OR (SEXO IN ('M', 'I') AND TPMORTEOCO IN ('1', '2', '3', '4', '5'))
```

In [24]:
# --- APLICAR FILTRO DE MORTALIDADE MATERNA ---
print("=" * 60)
print("FILTRO DE MORTALIDADE MATERNA")
print("=" * 60)

# Aplicar o filtro usando sexo_normalizado e IDADE como string
# e TPMORTEOCO como string (dados lidos como str)
mask_feminino_fertil = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
)

mask_excecao_mi = (
    (df_sim['sexo_normalizado'].isin(['M', 'I'])) &
    (df_sim['TPMORTEOCO'].isin(['1', '2', '3', '4', '5']))
)

total_geral = len(df_sim)
total_filtro = mask_feminino_fertil.sum() + mask_excecao_mi.sum()

print(f"Total de registros no dataset completo: {total_geral}")
print(f"")
print(f"--- Filtro principal (F, 10-49 anos): {mask_feminino_fertil.sum():,} ({mask_feminino_fertil.sum()/total_geral*100:.1f}%)")
print(f"--- Exceção (M/I com TPMORTEOCO): {mask_excecao_mi.sum():,} ({mask_excecao_mi.sum()/total_geral*100:.1f}%)")
print(f"")
print(f"Total de registros elegíveis para a fato: {total_filtro:,} ({total_filtro/total_geral*100:.1f}%)")

# Detalhar a exceção
if mask_excecao_mi.sum() > 0:
    print(f"\n--- Detalhamento da exceção M/I ---")
    excecao = df_sim[mask_excecao_mi]
    print(f"Por SEXO normalizado:")
    print(excecao['sexo_normalizado'].value_counts().to_string())
    print(f"\nPor TPMORTEOCO:")
    print(excecao['TPMORTEOCO'].value_counts().to_string())

FILTRO DE MORTALIDADE MATERNA
Total de registros no dataset completo: 14179520

--- Filtro principal (F, 10-49 anos): 700,464 (4.9%)
--- Exceção (M/I com TPMORTEOCO): 11 (0.0%)

Total de registros elegíveis para a fato: 700,475 (4.9%)

--- Detalhamento da exceção M/I ---
Por SEXO normalizado:
sexo_normalizado
I    10
M     1

Por TPMORTEOCO:
TPMORTEOCO
1    4
2    3
4    2
3    2


## 16. RESUMO — Valores Distintos por Variável

Tabela consolidada para validação do DDL.

In [28]:
# Resumo consolidado
print("=" * 80)
print("RESUMO - Valores Distintos por Variável para o Modelo Dimensional")
print("=" * 80)

variaveis = [
    ('SEXO', 'Atributo direto', df_sim['SEXO'].dropna().unique().tolist()),
    ('IDADE (1º dígito)', 'Dim_FaixaEtaria', df_sim['IDADE'].dropna().str[0].unique().tolist()),
    ('RACACOR', 'Dim_RacaCor', sorted(df_sim['RACACOR'].dropna().unique())),
    ('LOCOCOR', 'Dim_LocalOcorrencia', sorted(df_sim['LOCOCOR'].dropna().unique())),
    ('TPMORTEOCO', 'Dim_SituacaoGestacional', sorted(df_sim['TPMORTEOCO'].dropna().unique())),
    ('PARTO', 'Dim_TipoParto', sorted(df_sim['PARTO'].dropna().unique())),
    ('OBITOPARTO', 'Dim_MomentoObitoParto', sorted(df_sim['OBITOPARTO'].dropna().unique())),
    ('GRAVIDEZ', 'Dim_TipoGravidez', sorted(df_sim['GRAVIDEZ'].dropna().unique())),
    ('SEMAGESTAC', 'Dim_SemanaGestacao', sorted(df_sim['SEMAGESTAC'].dropna().unique())),
    ('ASSISTMED', 'Atributo direto', sorted(df_sim['ASSISTMED'].dropna().unique())),
]

for var, dim, valores in variaveis:
    nulos = df_sim[var].isna().sum() if var in df_sim.columns else 0
    print(f"\n{var:20s} → {dim:25s} | Nulos: {nulos:>8,} | Distintos: {len(valores):>4} | {valores}")

print(f"\n\n--- CID ---")
for col in ['CAUSABAS', 'CAUSAMAT', 'CAUSABAS_O']:
    n_cids = df_sim[col].nunique(dropna=False)
    nulos = df_sim[col].isna().sum()
    print(f"{col:20s} → Dim_CID (role-playing) | Nulos: {nulos:>8,} | CIDs distintos: {n_cids:>6,}")

print(f"\n--- Municípios ---")
for col in ['CODMUNOCOR', 'CODMUNRES']:
    n_muns = df_sim[col].nunique(dropna=False)
    nulos = df_sim[col].isna().sum()
    print(f"{col:20s} → Dim_Municipio (role-playing) | Nulos: {nulos:>8,} | Municípios distintos: {n_muns:>6,}")

print(f"\n--- Estabelecimentos ---")
n_estab = df_sim['CODESTAB'].nunique(dropna=False)
nulos_estab = df_sim['CODESTAB'].isna().sum()
print(f"{'CODESTAB':20s} → Dim_EstabelecimentoSaude       | Nulos: {nulos_estab:>8,} | Estabelecimentos distintos: {n_estab:>6,}")

print(f"\n--- Datas ---")
# DTOBITO no formato DDMMYYYY - extrair ano das posições 4-7
anos_dtobito = sorted(df_sim['DTOBITO'].dropna().str[4:8].unique())
print(f"{'DTOBITO':20s} → Dim_Tempo                       | Nulos: {df_sim['DTOBITO'].isna().sum():>8,} | Datas distintas: {df_sim['DTOBITO'].nunique():>6,} | Anos: {anos_dtobito}")

RESUMO - Valores Distintos por Variável para o Modelo Dimensional

SEXO                 → Atributo direto           | Nulos:        0 | Distintos:    3 | ['1', '2', '0']

IDADE (1º dígito)    → Dim_FaixaEtaria           | Nulos:        0 | Distintos:    7 | ['4', '2', '5', '0', '1', '3', '9']

RACACOR              → Dim_RacaCor               | Nulos:  416,826 | Distintos:    6 | ['1', '2', '3', '4', '5', '9']

LOCOCOR              → Dim_LocalOcorrencia       | Nulos:        3 | Distintos:    7 | ['1', '2', '3', '4', '5', '6', '9']

TPMORTEOCO           → Dim_SituacaoGestacional   | Nulos: 13,267,837 | Distintos:    9 | ['1', '2', '3', '4', '5', '6', '7', '8', '9']

PARTO                → Dim_TipoParto             | Nulos: 13,859,959 | Distintos:    3 | ['1', '2', '9']

OBITOPARTO           → Dim_MomentoObitoParto     | Nulos: 13,862,603 | Distintos:    4 | ['1', '2', '3', '9']

GRAVIDEZ             → Dim_TipoGravidez          | Nulos: 13,858,354 | Distintos:    4 | ['1', '2', '3', '9']

In [26]:
# Consolidado final: verificar compatibilidade dos valores com o DDL
print("=" * 80)
print("VALIDAÇÃO DE COMPATIBILIDADE COM O DDL")
print("=" * 80)

# Regras de validação
validacoes = [
    ("SEXO", "Deve conter apenas {'F','M','I', nulo}", 
     set(df_sim['SEXO'].dropna().unique()) <= {'F', 'M', 'I', '1', '2', '0', '9'}),
    ("RACACOR", "Deve conter apenas {'1','2','3','4','5', nulo}",
     set(df_sim['RACACOR'].dropna().unique()) <= {'1', '2', '3', '4', '5'}),
    ("LOCOCOR", "Deve conter apenas {'1','2','3','4','5','6','9', nulo}",
     set(df_sim['LOCOCOR'].dropna().unique()) <= {'1', '2', '3', '4', '5', '6', '9'}),
    ("TPMORTEOCO", "Deve conter apenas {'1','2','3','4','5','8','9', nulo}",
     set(df_sim['TPMORTEOCO'].dropna().unique()) <= {'1', '2', '3', '4', '5', '8', '9'}),
    ("PARTO", "Deve conter apenas {'1','2','9', nulo}",
     set(df_sim['PARTO'].dropna().unique()) <= {'1', '2', '9'}),
    ("OBITOPARTO", "Deve conter apenas {'1','2','3','9', nulo}",
     set(df_sim['OBITOPARTO'].dropna().unique()) <= {'1', '2', '3', '9'}),
    ("GRAVIDEZ", "Deve conter apenas {'1','2','3','9', nulo}",
     set(df_sim['GRAVIDEZ'].dropna().unique()) <= {'1', '2', '3', '9'}),
    ("CAUSABAS", "Não deve conter nulos (NOT NULL na fato)",
     df_sim['CAUSABAS'].notna().all()),
    ("CODMUNOCOR", "Não deve conter nulos (NOT NULL na fato)",
     df_sim['CODMUNOCOR'].notna().all()),
]

print(f"{'Variável':20s} {'Esperado':<45s} {'Status':<10s}")
print("-" * 75)
for var, esperado, resultado in validacoes:
    status = "✓ OK" if resultado else "✗ PROBLEMA"
    print(f"{var:20s} {esperado:<45s} {status:<10s}")

VALIDAÇÃO DE COMPATIBILIDADE COM O DDL
Variável             Esperado                                      Status    
---------------------------------------------------------------------------
SEXO                 Deve conter apenas {'F','M','I', nulo}        ✓ OK      
RACACOR              Deve conter apenas {'1','2','3','4','5', nulo} ✗ PROBLEMA
LOCOCOR              Deve conter apenas {'1','2','3','4','5','6','9', nulo} ✓ OK      
TPMORTEOCO           Deve conter apenas {'1','2','3','4','5','8','9', nulo} ✗ PROBLEMA
PARTO                Deve conter apenas {'1','2','9', nulo}        ✓ OK      
OBITOPARTO           Deve conter apenas {'1','2','3','9', nulo}    ✓ OK      
GRAVIDEZ             Deve conter apenas {'1','2','3','9', nulo}    ✓ OK      
CAUSABAS             Não deve conter nulos (NOT NULL na fato)      ✓ OK      
CODMUNOCOR           Não deve conter nulos (NOT NULL na fato)      ✓ OK      
